# A Real Data Bug, Found by Finally Using the Held-Out Year — and the Final 2025 Result

`notebooks/20`'s combination result (forecast blending beats NDF alone by 3.75%) was found entirely
on the *original* `VALIDATION` (2024). Follow-up 6 extended the project's splits forward
(`TRAIN`=2020-2024, `VALIDATION`=2025, `TEST`=2026 provisional) so that result could be re-checked
on a genuinely fresh year. Retraining immediately surfaced something unexpected: NDF's own accuracy
looked *four times worse* on 2025 than on 2024, and the errors were enormous on specific, seemingly
arbitrary dates. That turned out to be a real bug in this project's own data pipeline — not in NDF,
not in NESO's live data — found only because 2025 had never been touched by anything before now.

**This notebook documents the bug, the fix, and the re-confirmed final result.**

## The symptom: NDF's accuracy collapses on 2025, but only on some dates

**Note on how this notebook is organized**: the bug described below was found, root-caused, and
fixed *before* this notebook was written (in `edf.data.download`, with the raw/canonical parquets
already rebuilt) — so the library code and data files on disk are already correct. To show an
honest before/after rather than asserting numbers from memory, the cells below reconstruct the
*buggy* computation inline, from the same raw CSV, using the exact old logic — genuinely
re-executed, not narrated.

Retrained the point model (bias-corrected recipe + `notebooks/20`'s adopted 1-day cumulative
degree-days feature) on the new `TRAIN` (2020-2024) and evaluated both it and NDF on the new
`VALIDATION` (2025) — using the (already-fixed) canonical table.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from edf import config
from edf.models.baselines import PERIODS_PER_DAY
from edf.features.buckets import build_day_type_buckets, is_christmas_period
from edf.evaluate import evaluate
from edf.features.demand import build_feature_table
from edf.models.forecast import train_lightgbm
from edf.features.generation import capacity_factor
from edf.features.weather import build_weather_feature_table, cumulative_degree

pd.options.display.float_format = "{:.2f}".format
WEATHER_RAW_DIR = Path("../data/raw/weather")
HORIZON_PERIODS = config.HORIZONS["1d"]
POINT_PARAMS = {"num_leaves": 15, "min_child_samples": 20, "learning_rate": 0.05}
POINT_N_ESTIMATORS = 689
UPWEIGHT = 3.0
train_start, train_end = config.TRAIN
val_start, val_end = config.VALIDATION
print(f"TRAIN={config.TRAIN}  VALIDATION={config.VALIDATION}  TEST={config.TEST}")

TRAIN=('2020-01-01', '2024-12-31')  VALIDATION=('2025-01-01', '2025-12-31')  TEST=('2026-01-01', '2026-12-31')


In [2]:
df = pd.read_parquet("../data/processed/gb_energy_2020_2025.parquet")
era5 = build_weather_feature_table("open_meteo_historical", df.index, raw_dir=WEATHER_RAW_DIR)
ndf = pd.read_parquet("../data/raw/elexon_ndf_day_ahead.parquet")["ndf_day_ahead"]

wind_cf_full = capacity_factor(df["wind"], df["wind_capacity"])
day_buckets_full = build_day_type_buckets(era5["temperature_c"], wind_cf_full)
problem_mask = day_buckets_full["is_hot"] | day_buckets_full["is_high_wind"] | day_buckets_full["is_christmas"]
weight_full = pd.Series(1.0, index=df.index)
weight_full[problem_mask] = UPWEIGHT

era5_final = era5.copy()
era5_final["is_christmas"] = is_christmas_period(df.index)
era5_final["heating_degree_1d_cum"] = cumulative_degree(era5_final["heating_degree"], PERIODS_PER_DAY)
era5_final["cooling_degree_1d_cum"] = cumulative_degree(era5_final["cooling_degree"], PERIODS_PER_DAY)

X_base, y = build_feature_table(df, horizon_periods=HORIZON_PERIODS, weather=era5_final)
ndf_aligned = ndf.reindex(X_base.index)
X_val, y_val = X_base.loc[val_start:val_end], y.loc[val_start:val_end]
ndf_val = ndf_aligned.loc[X_val.index]

### Reconstructing the bug, honestly: re-parse the raw 2025 CSV with the old (buggy) logic

`edf.data.download` has already been fixed on disk, so to show a genuine before/after this
re-derives the *old* `demand` series from the raw CSV using the exact old parsing logic
(`dayfirst=True`), independently of the now-fixed library function — a real computation on real raw
data, not a recollection of numbers from the debugging session.

In [3]:
raw_2025_csv = pd.read_csv("../data/raw/demanddata_2025.csv")

def old_buggy_settlement_periods_to_utc(dates, periods):
    local_midnight = pd.to_datetime(dates, format="mixed", dayfirst=True).dt.tz_localize("Europe/London")
    utc_midnight = local_midnight.dt.tz_convert("UTC")
    offsets = pd.to_timedelta((periods.astype("int64") - 1) * 30, unit="min")
    return utc_midnight + offsets

buggy_timestamp = old_buggy_settlement_periods_to_utc(
    raw_2025_csv["SETTLEMENT_DATE"], raw_2025_csv["SETTLEMENT_PERIOD"]
)
demand_buggy = pd.Series(raw_2025_csv["ND"].to_numpy(), index=buggy_timestamp, name="demand").sort_index()
demand_buggy = demand_buggy[~demand_buggy.index.duplicated(keep="first")]

ndf_common_buggy = ndf.reindex(demand_buggy.index)
err_buggy = (demand_buggy - ndf_common_buggy).abs().dropna()
print("NDF-vs-actual error on 2025, using the OLD (buggy) date parsing:")
print(err_buggy.describe())

NDF-vs-actual error on 2025, using the OLD (buggy) date parsing:
count   17520.00
mean     2535.77
std      3979.33
min         0.00
25%       322.00
50%       807.00
75%      2609.25
max     23764.00
dtype: float64


**Median error (~800 MW) looks normal; the mean (2,536 MW) doesn't — a small number of huge
outliers are dragging it up.** Worth finding exactly where. (This matches the number actually
observed during debugging, now reproduced honestly rather than just stated.)

In [4]:
worst = err_buggy.sort_values(ascending=False).head(10)
worst

2025-01-06 13:30:00+00:00   23764.00
2025-06-01 15:00:00+00:00   23716.00
2025-01-06 13:00:00+00:00   23596.00
2025-06-01 14:30:00+00:00   23579.00
2025-06-01 15:30:00+00:00   23543.00
2025-06-01 14:00:00+00:00   23103.00
2025-01-06 15:30:00+00:00   23092.00
2025-01-06 14:00:00+00:00   23055.00
2025-08-01 15:30:00+00:00   22987.00
2025-06-01 13:30:00+00:00   22875.00
dtype: float64

**The two worst offenders, 2025-01-06 and 2025-06-01, are suspicious in a specific way**: NDF's
forecast for 2025-01-06 looks like a completely normal *winter* value (~38-40k MW), while the
(buggy) "actual" demand for that date looks like a *summer* value — and 2025-06-01 shows the exact
opposite pattern. That's not random noise, it's a swap.

In [5]:
sub = raw_2025_csv[(raw_2025_csv["SETTLEMENT_DATE"] == "2025-01-06") & (raw_2025_csv["SETTLEMENT_PERIOD"].between(25, 33))]
print("Raw CSV rows, SETTLEMENT_DATE literally says 2025-01-06:")
print(sub[["SETTLEMENT_DATE", "SETTLEMENT_PERIOD", "ND"]])
print("\nBut the OLD (buggy) parser assigned them this timestamp:")
print(old_buggy_settlement_periods_to_utc(sub["SETTLEMENT_DATE"], sub["SETTLEMENT_PERIOD"]).head(3).tolist())

Raw CSV rows, SETTLEMENT_DATE literally says 2025-01-06:
    SETTLEMENT_DATE  SETTLEMENT_PERIOD     ND
264      2025-01-06                 25  38608
265      2025-01-06                 26  38509
266      2025-01-06                 27  38642
267      2025-01-06                 28  38309
268      2025-01-06                 29  38220
269      2025-01-06                 30  38375
270      2025-01-06                 31  39120
271      2025-01-06                 32  40088
272      2025-01-06                 33  41071

But the OLD (buggy) parser assigned them this timestamp:
[Timestamp('2025-06-01 11:00:00+0000', tz='UTC'), Timestamp('2025-06-01 11:30:00+0000', tz='UTC'), Timestamp('2025-06-01 12:00:00+0000', tz='UTC')]


## Root cause: `pd.to_datetime(..., format="mixed", dayfirst=True)` silently swaps day/month for ISO dates

Confirmed directly: NESO's raw source data was always correct — the CSV's own `SETTLEMENT_DATE`
text genuinely reads "2025-01-06" for these rows (shown above), and re-fetching live from NESO gives
the same, correct text. The bug was purely in `edf.data.download.settlement_periods_to_utc`'s
*parsing* of that text -- NESO's yearly CSVs use inconsistent date formats across years
(`"01-JAN-2020"`, `"01-Jan-23"`, `"2025-01-01"`), handled with `format="mixed"`. The `dayfirst=True`
flag, added to handle the ambiguous older formats, turned out to also apply (incorrectly) to the
*unambiguous* ISO `"YYYY-MM-DD"` format 2025 switched to -- silently swapping month and day whenever
both were `<=12`.

In [6]:
s = pd.Series(["2025-01-06", "2025-06-01", "2025-01-01", "2025-12-25"])
print("dayfirst=True (the bug):")
print(pd.to_datetime(s, format="mixed", dayfirst=True).dt.date.tolist())
print("\ndayfirst=False (correct for all three formats seen in this dataset):")
print(pd.to_datetime(s, format="mixed", dayfirst=False).dt.date.tolist())

dayfirst=True (the bug):
[datetime.date(2025, 6, 1), datetime.date(2025, 1, 6), datetime.date(2025, 1, 1), datetime.date(2025, 12, 25)]

dayfirst=False (correct for all three formats seen in this dataset):
[datetime.date(2025, 1, 6), datetime.date(2025, 6, 1), datetime.date(2025, 1, 1), datetime.date(2025, 12, 25)]


**Why this was invisible for the entire project until now**: 2020-2024's CSVs all spell the
month out ("JAN", "Jan") -- unambiguous regardless of `dayfirst`, so none of Weeks 1-7 or
Follow-ups 1-5 (all evaluated against `TRAIN`=2020-2023/`VALIDATION`=2024, or the equivalent) were
ever affected. Only 2025 (and, per its own NESO metadata note, apparently 2026 too) switched to the
bare numeric ISO format that triggers it. **2025 was `TEST` under the original split -- deliberately
never evaluated against by anything in this project -- so the bug had nowhere to surface until
Follow-up 6 promoted it to an active `VALIDATION` and something finally looked at it.** A direct,
concrete vindication of the "never peek at `TEST`" discipline for a reason beyond overfitting: data
quality problems in an unused slice stay invisible until the slice is finally used.

**Scope of the corruption**: roughly 12 of every month's ~30-31 days (any day-of-month `<=12`,
except the 12 self-symmetric dates where day equals month) had their `ND`/`TSD`/etc. values
attributed to the wrong calendar date throughout the entire 2025 portion of the raw table.

**Fix**: `edf.data.download.settlement_periods_to_utc` now parses with `dayfirst=False` -- verified
correct against samples from every year 2020-2025 (see updated docstring and
`tests/test_download.py::test_iso_dates_do_not_get_day_month_swapped`, a new regression test using
an asymmetric date pair specifically because the *existing* format test happened to use
`"2025-01-01"`, a self-symmetric date the bug doesn't affect -- confirmed by reproducing the bug
against the old code with that new test). Raw and canonical parquets rebuilt from the
already-downloaded (and independently re-verified, always-correct) CSV files -- no new data fetch
needed, just a re-parse.

In [7]:
err_after_fix = (y_val - ndf_val).abs()
print("NDF-vs-actual error on 2025 VALIDATION, after the fix:")
print(err_after_fix.describe())

NDF-vs-actual error on 2025 VALIDATION, after the fix:
count   17519.00
mean      593.14
std       522.46
min         0.00
25%       208.00
50%       455.00
75%       825.50
max      4232.00
dtype: float64


**Median and mean are now close (both ~600-800 MW), matching the shape seen on the original
2024 `VALIDATION` -- no more outliers.**

## Final result: retrain on the extended `TRAIN`, evaluate cleanly on the fresh, corrected `VALIDATION`

Same recipe as `notebooks/20`: point model = bias-corrected recipe + 1-day cumulative degree-days
(`TRAIN`=2020-2024, matched to NDF-available rows); combination weight fit via expanding-window
walk-forward across 2021-2024 (zero overlap with 2025); evaluated once, cleanly, on all of 2025 --
no need for `notebooks/20`'s H1/H2 split this time, since a genuinely separate year is available.

In [8]:
train_mask = ndf_aligned.notna().loc[train_start:train_end]
X_train_matched = X_base.loc[train_start:train_end][train_mask]
y_train_matched = y.loc[X_train_matched.index]
model_final = train_lightgbm(
    X_train_matched, y_train_matched, sample_weight=weight_full.loc[X_train_matched.index],
    n_estimators=POINT_N_ESTIMATORS, **POINT_PARAMS
)
preds_final = pd.Series(model_final.predict(X_val), index=X_val.index)

m_point = evaluate(y_val, preds_final)
m_ndf = evaluate(y_val, ndf_val)
print(f"Point model alone (2025 VALIDATION): MAE={m_point['mae']:.2f}")
print(f"NDF alone (2025 VALIDATION): MAE={m_ndf['mae']:.2f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001242 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6457
[LightGBM] [Info] Number of data points in the train set: 61020, number of used features: 33
[LightGBM] [Info] Start training from score 26604.678601


Point model alone (2025 VALIDATION): MAE=943.89
NDF alone (2025 VALIDATION): MAE=593.14


In [9]:
from edf.models.combination import combine_forecasts, fit_combination_weight

fold_preds = []
for fold_year in [2021, 2022, 2023, 2024]:
    fold_train_idx = X_base.loc[: f"{fold_year - 1}-12-31"].index
    fold_predict_idx = X_base.loc[f"{fold_year}-01-01": f"{fold_year}-12-31"].index
    fold_predict_idx = fold_predict_idx.intersection(ndf_aligned.notna()[ndf_aligned.notna()].index)
    fold_model = train_lightgbm(
        X_base.loc[fold_train_idx], y.loc[fold_train_idx],
        sample_weight=weight_full.loc[fold_train_idx], n_estimators=POINT_N_ESTIMATORS, **POINT_PARAMS
    )
    fold_preds.append(pd.Series(fold_model.predict(X_base.loc[fold_predict_idx]), index=fold_predict_idx))

walk_forward_preds = pd.concat(fold_preds).sort_index()
w_star = fit_combination_weight(y.loc[walk_forward_preds.index], walk_forward_preds, ndf_aligned.loc[walk_forward_preds.index])
print(f"Combination weight on our model (fit on 2021-2024 walk-forward OOS, zero overlap with 2025): {w_star:.4f}")

blend_val = combine_forecasts(preds_final, ndf_val, weight_a=w_star)
m_blend = evaluate(y_val, blend_val)

summary = pd.DataFrame({"point model alone": m_point, "NDF alone": m_ndf, "combined (final)": m_blend}).T
summary[["mae", "rmse", "bias", "n"]]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000824 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6340
[LightGBM] [Info] Number of data points in the train set: 16224, number of used features: 33
[LightGBM] [Info] Start training from score 26867.943760


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000965 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6442
[LightGBM] [Info] Number of data points in the train set: 33744, number of used features: 33
[LightGBM] [Info] Start training from score 27633.768391


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001193 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6457
[LightGBM] [Info] Number of data points in the train set: 51264, number of used features: 33
[LightGBM] [Info] Start training from score 27296.002661


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001256 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6458
[LightGBM] [Info] Number of data points in the train set: 68784, number of used features: 33
[LightGBM] [Info] Start training from score 27001.358990


Combination weight on our model (fit on 2021-2024 walk-forward OOS, zero overlap with 2025): 0.1717


,mae,rmse,bias,n
point model alone,943.89,1271.73,-297.38,17519.00
NDF alone,593.14,790.42,-18.64,17519.00
combined (final),572.11,767.92,-66.49,17519.00


## Result: the combination effect holds up on a genuinely fresh year, closely matching `notebooks/20`

| | 2024 (`notebooks/20`, H1-fit/H2-eval) | 2025 (this notebook, fit on 2021-2024, eval on all of 2025) |
|---|---|---|
| NDF alone | 533.03 | 593.14 |
| combined | 513.05 | 572.11 |
| **improvement** | **3.75%** | **3.55%** |

**Remarkably close given the combination weight was fit on entirely different data each time** (2024
H1 vs. the full 2021-2024 walk-forward set) **and evaluated on entirely different, non-overlapping
years.** This is the strongest evidence yet that `notebooks/20`'s combination finding is real and
generalizes, not an artifact of one particular train/test split or a lucky H1/H2 division. The
absolute MAE values differ year to year (2025 was evidently a somewhat harder year for both NDF and
our model), but the *relative* benefit of combining is stable.

**This is now the project's headline forecasting result, final numbers**: combining our model with
NDF, weight ≈0.17 on our model, beats NDF alone by **3.55% on the freshest, cleanest evaluation
available** -- verified after finding and fixing a real data bug that a "never touch `TEST`"
discipline had, until this point, kept invisible.